# Phase 4 - Pandas: `read_csv()` Deep Dive + Core EDA Notes

Detailed, hands-on reference for `pd.read_csv()` and the small set of pandas
operations you will use in almost **every** ML / Data Science project.

Each numbered point below has:
- **What** the parameter/operation does
- **Why** you'd actually use it (real project scenario)
- **Code** you can run directly on the Titanic `train.csv` dataset

> Dataset used throughout: Titanic `train.csv`. Update `CSV_PATH` below if your
> file lives somewhere else.

In [4]:
from xxlimited_35 import error

import pandas as pd
import numpy as np
import io

CSV_PATH = r'/assets/data/titanic.csv'

## 1. Basic `read_csv()` - default behaviour

- Reading a CSV with **no extra parameters** loads every row and column
  exactly as pandas infers them.
- Pandas always creates its own default **RangeIndex (0, 1, 2, ...)** even
  if the CSV already has an ID-like column (e.g. `PassengerId`).
- This is the very first thing to check on any new dataset: shape, columns,
  dtypes, first few rows.

**Use case:** quick first look at any new dataset before deciding what to
clean or keep.

In [5]:
df = pd.read_csv(CSV_PATH)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. `index_col` - use an existing column as the row index

- If the CSV already has a natural unique identifier (`PassengerId`,
  `CustomerID`, `OrderID`...), use it as the index instead of pandas'
  auto-generated one.
- Makes `.loc[]` lookups by that ID direct and intuitive.

**Use case:** any dataset with a primary-key-like column.

In [6]:
df = pd.read_csv(CSV_PATH, index_col='PassengerId')
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. `usecols` - load only the columns you need

- Real datasets often have far more columns than you actually need for a
  given analysis/model.
- `usecols` skips parsing the rest, which **saves memory and load time** -
  important on large files.

**Use case:** you already know which features you'll use for the model.

In [7]:
df = pd.read_csv(
    CSV_PATH,
    index_col='PassengerId',
    usecols=['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Fare', 'Embarked'],
)
df.head()

,Survived,Pclass,Name,Sex,Fare,Embarked
PassengerId,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,7.2500,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,71.2833,C
3,1,3,"Heikkinen, Miss. Laina",female,7.9250,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,53.1000,S
5,0,3,"Allen, Mr. William Henry",male,8.0500,S


## 4. `names` + `skiprows` - CSV with no header row

- Some CSVs (especially exports from other systems) don't ship column
  names at all - the first row is already data.
- `names=[...]` supplies your own column names.
- `skiprows=1` is used together with `names` **only when the file does
  actually contain a header line you want to discard** (otherwise omit
  `skiprows` and let `names` label row 0 as data).

**Use case:** headerless exports, or when you want to rename every column
in one shot instead of using `.rename()` afterwards.

In [8]:
df = pd.read_csv(
    CSV_PATH,
    names=['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l'],
    index_col='a',
    skiprows=1,  # the original header row becomes row 0, so skip it
)
df.head()

,b,c,d,e,f,g,h,i,j,k,l
a,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 5. `header` - tell pandas which row holds the column names

- By default `header='infer'` -> pandas assumes **row 0** is the header.
- If real headers sit a few rows down (common in messy exports with title
  rows/notes above the data), pass the row number: `header=1`, `header=2`...
- `header=None` means "there is no header at all" -> pandas assigns
  integer column names `0, 1, 2, ...`.

**Use case:** files exported with a report title/metadata above the real
table, or completely headerless files.

In [9]:
# Row index 1 (the 2nd line) becomes the header; everything above is discarded
df = pd.read_csv(CSV_PATH, header=1)
df.head()

,1,0,3,"Braund, Mr. Owen Harris",male,22,1.1,0.1,A/5 21171,7.25,Unnamed: 10,S
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
1,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
2,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
3,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
4,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q


In [10]:
# No header at all -> integer column names
df_no_header = pd.read_csv(CSV_PATH, header=None)
df_no_header.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11
0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.25,NaN,S
2,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C


## 6. `nrows` - load only the first N rows

- Great for **previewing huge files** without waiting to load millions of
  rows or blowing up memory.
- Common first step: `nrows=5` or `nrows=1000` to sanity-check dtypes and
  structure before doing the full load.

**Use case:** exploring a multi-GB CSV, or building/testing pipeline code
quickly on a small sample.

In [11]:
df = pd.read_csv(CSV_PATH, index_col='PassengerId', nrows=10)
df

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S


## 7. `skiprows` (rows to skip) - drop junk rows from the top

- Accepts an **integer** (skip the first N rows), a **list of row numbers**
  (skip specific rows), or a **callable** (skip rows where the function
  returns `True`).
- Very handy for files with junk/description lines before the real data.

**Use case:** skip every other row, skip specific known-bad rows, or skip a
metadata block at the top.

In [12]:
# Skip specific row numbers (0-indexed, counted before header handling)
df = pd.read_csv(CSV_PATH, skiprows=[1, 3, 5])
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
1,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
2,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
3,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
4,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S


In [13]:
# Skip using a function: here, skip every odd-numbered data row
df = pd.read_csv(CSV_PATH, skiprows=lambda row_idx: row_idx > 0 and row_idx % 2 == 0)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
2,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
3,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
4,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S


## 8. `on_bad_lines` - handle malformed rows (extra/missing fields)

- Older pandas used `error_bad_lines` / `warn_bad_lines`. **Both are
  deprecated (pandas 1.3) and removed (pandas 2.0+)**.
- Modern replacement: `on_bad_lines='error' | 'warn' | 'skip'`
  (default is `'error'`, which raises on malformed rows).
- `on_bad_lines='skip'` silently drops broken rows; `'warn'` drops them but
  prints a warning so you know data was lost.

**Use case:** messy real-world CSVs where a handful of rows have too many
or too few commas (e.g. an unescaped comma inside a text field).

In [14]:
df = pd.read_csv(
    CSV_PATH,
    engine='python',      # on_bad_lines='warn' needs the python engine
    on_bad_lines='warn',  # 'skip' to silently drop, 'error' to raise (default)
)
df.shape

(891, 12)

## 9. `dtype` - set column data types explicitly

- Pandas guesses dtypes by scanning the file, which can be **slow** and
  sometimes **wrong** (e.g. an ID column read as `int64` when you want
  `str` so leading zeros aren't lost).
- Passing `dtype={...}` up front is faster and safer than converting
  columns after loading.

**Use case:** ID/code columns that must stay as strings, or forcing a
numeric column to a smaller/faster dtype like `int32`/`float32`.

In [15]:
df = pd.read_csv(
    CSV_PATH,
    dtype={'PassengerId': str, 'Pclass': 'int8', 'Fare': 'float32'},
)
df.dtypes

PassengerId     object
Survived         int64
Pclass            int8
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float32
Cabin           object
Embarked        object
dtype: object

## 10. `parse_dates` - auto-convert date columns to `datetime64`

- Any column that looks like a date but loads as plain text (`object`
  dtype) should be parsed with `parse_dates=[...]` so you can use
  `.dt` accessors, resampling, date arithmetic, etc.
- Titanic's `train.csv` has no date column, so here's a tiny inline
  example to show the behaviour clearly.

**Use case:** any time-series or log data - signup dates, order dates,
timestamps.

In [16]:
sample_csv = io.StringIO('id,signup_date,amount\n1,2024-01-05,100\n2,2024-02-14,250\n')

df_dates = pd.read_csv(sample_csv, parse_dates=['signup_date'])
print(df_dates.dtypes)
df_dates

id                      int64
signup_date    datetime64[ns]
amount                  int64
dtype: object


,id,signup_date,amount
0,1,2024-01-05,100
1,2,2024-02-14,250


## 11. `na_values` - tell pandas what "missing" looks like in this file

- Pandas recognises common blanks as `NaN` by default, but real data often
  uses placeholders like `'?'`, `'-'`, `'N/A'`, `'Unknown'`, `999`.
- `na_values=[...]` (globally, or per-column via a dict) makes those
  values show up correctly as missing instead of a weird string/number.

**Use case:** any dataset with custom missing-value codes - very common in
survey and sensor data.

In [17]:
df = pd.read_csv(
    CSV_PATH,
    na_values=['?', '-', 'Unknown', ''],
)
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## 12. `sep` / `delimiter` - non-comma separated files

- CSV doesn't always mean *comma*-separated: tab (`\t`), semicolon (`;`)
  and pipe (`|`) are all common, especially in European exports.
- `sep=None, engine='python'` auto-detects the delimiter (slower, but
  useful when you don't know it up front).

**Use case:** `.tsv` files, semicolon-delimited European CSVs, pipe-
delimited log exports.

In [18]:
tsv_sample = io.StringIO('name\tage\tcity\nAmit\t28\tDelhi\nSara\t34\tMumbai\n')
df_tsv = pd.read_csv(tsv_sample, sep='\t')
df_tsv

,name,age,city
0,Amit,28,Delhi
1,Sara,34,Mumbai


## 13. `encoding` - fixing `UnicodeDecodeError`

- Default encoding is `utf-8`. Files from Excel/Windows are frequently
  saved as `latin-1` / `cp1252` instead, which crashes a plain `read_csv`.
- Fix: pass `encoding='latin-1'` (or `'cp1252'`), or use
  `encoding_errors='replace'` to substitute unreadable characters instead
  of crashing.

**Use case:** any `UnicodeDecodeError` when reading a client-provided CSV.

In [19]:
# df = pd.read_csv(CSV_PATH, encoding='utf-8')          # default, works here
# df = pd.read_csv(CSV_PATH, encoding='latin-1')        # try this if utf-8 fails
# df = pd.read_csv(CSV_PATH, encoding_errors='replace') # never crash, replace bad chars
df = pd.read_csv(CSV_PATH, encoding='utf-8')
df.shape

(891, 12)

## 14. `converters` - transform values while reading

- A dict mapping column name -> function, applied to every raw value in
  that column **as it is read** (before dtype inference).
- Cheaper than loading then calling `.apply()` afterwards on huge files.

**Use case:** cleaning a `'67%'` string into `0.67`, standardising casing,
stripping currency symbols, etc.

In [20]:
df = pd.read_csv(
    CSV_PATH,
    converters={'Sex': lambda s: s.strip().upper()},
)
df['Sex'].unique()

array(['MALE', 'FEMALE'], dtype=object)

## 15. `chunksize` - reading files too big for memory

- Returns an **iterator of DataFrames** ('chunks') instead of one giant
  DataFrame, so you can process a file piece by piece.
- Classic pattern: loop over chunks, aggregate as you go (sum, filter,
  write to a database), and never hold the whole file in RAM.

**Use case:** multi-GB CSVs that don't fit in memory.

In [21]:
total_fare = 0.0
row_count = 0

for chunk in pd.read_csv(CSV_PATH, usecols=['Fare'], chunksize=200):
    total_fare += chunk['Fare'].sum()
    row_count += len(chunk)

print(f'Rows processed: {row_count}, Total fare: {total_fare:.2f}')

Rows processed: 891, Total fare: 28693.95


## 16. `compression` - read compressed CSVs directly

- Pandas can read `.csv.gz`, `.csv.zip`, `.csv.bz2`, `.csv.xz` without you
  manually decompressing first.
- `compression='infer'` (the default) figures it out from the file
  extension automatically.

**Use case:** datasets shared as gzip/zip archives (very common on
Kaggle/data portals) - saves an extra unzip step.

In [22]:
# Example (uncomment once you have a compressed file):
# df = pd.read_csv('titanic.csv.gz', compression='infer')
# df = pd.read_csv('titanic.csv.zip', compression='zip')
print('compression="infer" auto-detects .gz / .zip / .bz2 / .xz from the filename')

compression="infer" auto-detects .gz / .zip / .bz2 / .xz from the filename


## 17. Getting a `Series` back (the old `squeeze` parameter)

- Older pandas had `squeeze=True` to return a `Series` instead of a
  1-column `DataFrame`. **It was removed in pandas 2.0.**
- Modern equivalent: read normally, then call `.squeeze('columns')`, or
  just select the column directly - both are one-liners and clearer.

**Use case:** you only need a single column as a `Series` for something
like `.value_counts()` or plotting.

In [23]:
s = pd.read_csv(CSV_PATH, usecols=['Sex']).squeeze('columns')
print(type(s))
s.head()

<class 'pandas.core.series.Series'>


0      male
1    female
2    female
3    female
4      male
Name: Sex, dtype: object

## 18. `index_col=False` - stop pandas guessing the index wrong

- On malformed files (e.g. a trailing delimiter on every line), pandas can
  mistakenly use the **first column as the index**.
- `index_col=False` forces pandas to *not* do that, keeping every column
  as a regular column.

**Use case:** files with a stray trailing comma/delimiter on each row.

In [24]:
df = pd.read_csv(CSV_PATH, index_col=False)
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 19. `to_csv()` - writing a DataFrame back to disk

- The write-side counterpart of `read_csv()`.
- `index=False` is the setting you'll use almost every time - otherwise
  pandas writes its RangeIndex as an extra unwanted column.
- Other handy options: `sep=';'`, `columns=[...]` (subset), `mode='a'`
  (append instead of overwrite).

**Use case:** saving a cleaned dataset for the next notebook/phase.

In [25]:
df = pd.read_csv(CSV_PATH, index_col='PassengerId')
# df.to_csv('cleaned_titanic.csv', index=False)   # index=False = almost always what you want
print('to_csv(..., index=False) avoids writing an extra unnamed index column')

to_csv(..., index=False) avoids writing an extra unnamed index column


## 20. First-look exploration commands (run these on every new dataset)

- `.head()` / `.tail()` - peek at the first/last rows
- `.shape` - (rows, columns)
- `.info()` - dtypes, non-null counts, memory usage in one shot
- `.describe()` - quick stats for numeric columns (mean, std, quartiles)
- `.columns` / `.dtypes` - column names and their types

**Use case:** the standard first 60 seconds with any new dataset, before
any cleaning or modelling starts.

In [26]:
df = pd.read_csv(CSV_PATH, index_col='PassengerId')

print(df.shape)
df.info()

(891, 11)
<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    object 
 10  Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 83.5+ KB


In [27]:
df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## 21. Missing value check - `isnull().sum()`

- The very next step after loading: **how much data is actually missing,
  and in which columns**?
- Drives every later decision - drop the column, drop the rows, or impute
  (fill) with mean/median/mode.

**Use case:** mandatory step before any feature engineering or model
training.

In [28]:
df.isnull().sum().sort_values(ascending=False)

Cabin       687
Age         177
Embarked      2
Survived      0
Pclass        0
Name          0
Sex           0
SibSp         0
Parch         0
Ticket        0
Fare          0
dtype: int64

## 22. `value_counts()` - frequency counts for categorical columns

- Shows how many times each unique value appears - instantly reveals class
  imbalance, typos/inconsistent categories, and rare categories.
- `normalize=True` gives proportions instead of raw counts; `dropna=False`
  includes missing values in the count.

**Use case:** checking target balance (`Survived`), or scanning a
categorical feature (`Sex`, `Embarked`, `Pclass`) before encoding it.

In [29]:
print(df['Survived'].value_counts(normalize=True))
print()
print(df['Embarked'].value_counts(dropna=False))

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


## 23. Cheat-sheet recap - most-used `read_csv` parameters & tips

| Parameter | Purpose |
|---|---|
| `index_col` | use an existing column as the row index |
| `usecols` | load only the columns you need (faster, less memory) |
| `names` + `skiprows` | headerless files / rename every column at once |
| `header` | which row holds the column names |
| `nrows` | preview only the first N rows |
| `skiprows` | drop specific/callable rows from the top |
| `on_bad_lines` | handle malformed rows (`'skip'`/`'warn'`/`'error'`) |
| `dtype` | force column types up front (speed + correctness) |
| `parse_dates` | auto-convert date columns to `datetime64` |
| `na_values` | teach pandas custom missing-value markers |
| `sep` / `delimiter` | non-comma files (tsv, semicolon, pipe) |
| `encoding` | fix `UnicodeDecodeError` |
| `converters` | transform values while reading |
| `chunksize` | stream huge files instead of loading fully |
| `compression` | read `.gz` / `.zip` CSVs directly |

**Performance tip:** for large files, always combine `usecols` + `dtype`
(and `chunksize` if it still doesn't fit in memory) - this is the single
biggest speed/memory win available.

**Workflow tip:** immediately after loading -> `.info()`, `.describe()`,
`isnull().sum()`, then `value_counts()` on your categorical columns and
target. This four-step routine catches most data-quality issues before
you write a single line of feature engineering.

In [36]:
try:
    df = pd.read_csv(
        CSV_PATH,
        index_col='PassengerId',
        usecols=['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Fare', 'Embarked'],
        header=None,
        engine='python',
        on_bad_lines='warn',
    )
except Exception as e:
    print(e)

Usecols do not match columns, columns expected but not found: ['Survived', 'PassengerId', 'Name', 'Fare', 'Sex', 'Pclass', 'Embarked']


In [46]:
Survived = df[['Survived']]

In [47]:
Survived

,Survived
PassengerId,
1,0
2,1
3,1
4,1
5,0
...,...
887,0
888,1
889,0
